# Healthcare CSV ETL to PostgreSQL

This notebook reads `raw_data/healthcare_dataset.csv`, cleans and validates the records, removes exact duplicate rows after normalization, and loads the result into the `healthcare_records` table.

Start the local PostgreSQL container from the project root before running the notebook:

```bash
docker compose up -d postgres
```

The service listens on host port **5434** and persists data in a Docker volume. Install notebook dependencies with `pip install -r requirements.txt`. The compose-file credentials are for local development only; override `POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`, and `POSTGRES_PORT` in the environment when needed. The same variables configure this notebook.

In [6]:
import os
import re
from pathlib import Path

import pandas as pd
from sqlalchemy import URL, create_engine, text

CSV_PATH = Path(os.getenv("RAW_CSV_PATH", "raw_data/healthcare_dataset.csv"))
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = int(os.getenv("POSTGRES_PORT", "5434"))
DB_NAME = os.getenv("POSTGRES_DB", "healthcare")
DB_USER = os.getenv("POSTGRES_USER", "healthcare")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "healthcare_dev_password")
TABLE_NAME = "healthcare_records"

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH.resolve()}")

database_url = URL.create(
    "postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(database_url, pool_pre_ping=True)
print(f"Source: {CSV_PATH.resolve()}")
print(f"Target: {DB_HOST}:{DB_PORT}/{DB_NAME}.{TABLE_NAME}")

Source: /home/mmagni/Documents/dev/mine/healthcare-agent-poc/raw_data/healthcare_dataset.csv
Target: localhost:5434/healthcare.healthcare_records


## Extract and inspect

Read the CSV without silently coercing malformed fields. The header check makes unexpected source-schema changes explicit.

In [7]:
expected_columns = [
    "Name", "Age", "Gender", "Blood Type", "Medical Condition",
    "Date of Admission", "Doctor", "Hospital", "Insurance Provider",
    "Billing Amount", "Room Number", "Admission Type", "Discharge Date",
    "Medication", "Test Results",
]

raw_df = pd.read_csv(CSV_PATH)
if list(raw_df.columns) != expected_columns:
    raise ValueError(
        "Unexpected CSV schema. "
        f"Expected {expected_columns}; received {list(raw_df.columns)}"
    )

print(f"Extracted {len(raw_df):,} rows and {len(raw_df.columns)} columns")
display(raw_df.head())
display(raw_df.isna().sum().rename("missing_values").to_frame())

Extracted 55,500 rows and 15 columns


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


,missing_values
Name,0
Age,0
Gender,0
Blood Type,0
Medical Condition,0
Date of Admission,0
Doctor,0
Hospital,0
Insurance Provider,0
Billing Amount,0


## Transform and validate

Headers become lowercase snake_case; text is trimmed and internal whitespace is collapsed. Leading honorifics (`Mr.`, `Mrs.`, `Ms.`, `Miss`, and `Dr.`; case-insensitive and with an optional period) are removed from patient names. The first remaining token becomes `name`, and the remaining token(s) become `surname`; both are title-cased to normalize the source capitalization. Dates and numeric columns are parsed strictly. Exact duplicates after these normalizations are removed; no records are excluded for unusual billing values.

In [8]:
def to_snake_case(column: str) -> str:
    column = re.sub(r"[^0-9a-zA-Z]+", "_", column.strip())
    return column.strip("_").lower()


df = raw_df.rename(columns=to_snake_case).copy()

for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip().str.replace(r"\s+", " ", regex=True)

honorific_pattern = r"^(?:mr|mrs|ms|miss|dr)\.?\s+"
honorific_rows = df["name"].str.contains(honorific_pattern, case=False, regex=True).sum()
df["name"] = df["name"].str.replace(
    honorific_pattern, "", case=False, regex=True
)
name_parts = df["name"].str.split(" ", n=1, expand=True)
if name_parts.shape[1] != 2 or name_parts.isna().any().any():
    raise ValueError("Every patient name must contain a first name and surname")
df["surname"] = name_parts[1].str.title()
df["name"] = name_parts[0].str.title()
df["doctor"] = df["doctor"].str.title()

for column in ("age", "room_number"):
    df[column] = pd.to_numeric(df[column], errors="raise").astype("int64")

df["billing_amount"] = pd.to_numeric(df["billing_amount"], errors="raise").round(2)
for column in ("date_of_admission", "discharge_date"):
    df[column] = pd.to_datetime(
        df[column], format="%Y-%m-%d", errors="raise"
    ).dt.date

if df.isna().any().any():
    missing = df.isna().sum()
    raise ValueError(f"Null values found after transformation:\n{missing[missing > 0]}")
if not df["age"].between(0, 120).all():
    raise ValueError("Age outside the accepted range 0-120")
if (df["date_of_admission"] > df["discharge_date"]).any():
    raise ValueError("Found admissions discharged before their admission date")

negative_bills = int((df["billing_amount"] < 0).sum())
source_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
removed_duplicates = source_rows - len(df)

print(f"Rows before exact-deduplication: {source_rows:,}")
print(f"Leading honorifics removed: {honorific_rows:,}")
print(f"Exact duplicate rows removed: {removed_duplicates:,}")
print(f"Rows ready to load: {len(df):,}")
print(f"Negative billing amounts retained for review: {negative_bills:,}")
display(df.head())

Rows before exact-deduplication: 55,500
Leading honorifics removed: 1,114
Exact duplicate rows removed: 534
Rows ready to load: 54,966
Negative billing amounts retained for review: 108


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,surname
0,Bobby,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.28,328,Urgent,2024-02-02,Paracetamol,Normal,Jackson
1,Leslie,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.33,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,Terry
2,Danny,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.10,205,Emergency,2022-10-07,Aspirin,Normal,Smith
3,Andrew,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78,450,Elective,2020-12-18,Ibuprofen,Abnormal,Watts
4,Adrienne,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.32,458,Urgent,2022-10-09,Penicillin,Abnormal,Bell


## Load

The table is created with explicit PostgreSQL types. Each run replaces the table contents inside one transaction, so a failed load does not leave a partially refreshed table.

In [9]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS healthcare_records (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name TEXT NOT NULL,
    surname TEXT NOT NULL,
    age INTEGER NOT NULL,
    gender TEXT NOT NULL,
    blood_type TEXT NOT NULL,
    medical_condition TEXT NOT NULL,
    date_of_admission DATE NOT NULL,
    doctor TEXT NOT NULL,
    hospital TEXT NOT NULL,
    insurance_provider TEXT NOT NULL,
    billing_amount NUMERIC(12, 2) NOT NULL,
    room_number INTEGER NOT NULL,
    admission_type TEXT NOT NULL,
    discharge_date DATE NOT NULL,
    medication TEXT NOT NULL,
    test_results TEXT NOT NULL
)
"""

with engine.begin() as connection:
    connection.execute(text(create_table_sql))
    connection.execute(
        text("ALTER TABLE healthcare_records ADD COLUMN IF NOT EXISTS surname TEXT")
    )
    connection.execute(text("TRUNCATE TABLE healthcare_records RESTART IDENTITY"))
    df.to_sql(
        TABLE_NAME,
        con=connection,
        if_exists="append",
        index=False,
        chunksize=2_000,
        method="multi",
    )
    connection.execute(
        text("ALTER TABLE healthcare_records ALTER COLUMN surname SET NOT NULL")
    )

print(f"Loaded {len(df):,} rows into {TABLE_NAME}")

Loaded 54,966 rows into healthcare_records


## Verify the load

Compare the loaded count with the transformed dataframe and inspect a small sample. A mismatch raises an error instead of reporting a successful ETL.

In [10]:
with engine.connect() as connection:
    loaded_count = connection.execute(
        text("SELECT COUNT(*) FROM healthcare_records")
    ).scalar_one()
    sample = pd.read_sql(
        text("SELECT * FROM healthcare_records ORDER BY id LIMIT 5"), connection
    )

if loaded_count != len(df):
    raise RuntimeError(
        f"Load verification failed: expected {len(df):,}, found {loaded_count:,}"
    )

print(f"Verified {loaded_count:,} rows in PostgreSQL")
display(sample)
engine.dispose()

Verified 54,966 rows in PostgreSQL


,id,name,surname,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,1,Bobby,Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.28,328,Urgent,2024-02-02,Paracetamol,Normal
1,2,Leslie,Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.33,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,3,Danny,Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.10,205,Emergency,2022-10-07,Aspirin,Normal
3,4,Andrew,Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.78,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,5,Adrienne,Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.32,458,Urgent,2022-10-09,Penicillin,Abnormal
